In [31]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.interpolate import RegularGridInterpolator

Gyr2sec = 1e9 * 365 * 24 * 3600

MPBH_values = np.logspace(13, 20, 20)
rho_values = np.logspace(6, 9, 15)

results_dir = "../../blackhawk_v2.3/results/PBH_evolution"

def read_lifetime(path):
    with open(path, "r") as f:
        lines = f.readlines()
    last_t = None
    for line in lines[2:]:
        line = line.strip()
        if not line:
            continue
        last_t = float(line.split()[0])
    return last_t

lifetime = np.full((len(rho_values), len(MPBH_values)), np.nan)

for i, rho in enumerate(rho_values):
    for j, M in enumerate(MPBH_values):
        path = os.path.join(results_dir, f"M{M:.2e}", f"rho{rho:.2e}", "dts.txt")
        lifetime[i, j] = read_lifetime(path) / Gyr2sec

n_fine = 200

log_M_base = np.log10(MPBH_values)
log_rho_base = np.log10(rho_values)
log_life = np.log10(lifetime)

interp_func = RegularGridInterpolator(
    (log_rho_base, log_M_base),
    log_life,
    method="cubic",
    bounds_error=False,
    fill_value=None
)

rho_fine = np.logspace(log_rho_base.min(), log_rho_base.max(), n_fine)
M_fine = np.logspace(log_M_base.min(), log_M_base.max(), n_fine)
rho_fine_grid, M_fine_grid = np.meshgrid(rho_fine, M_fine)

eval_points = np.column_stack([np.log10(rho_fine_grid).ravel(), np.log10(M_fine_grid).ravel()])
log_life_fine = interp_func(eval_points).reshape(n_fine, n_fine)

lifetime_fine = 10.**log_life_fine

fig, ax = plt.subplots(figsize=(8, 6))

norm = LogNorm(vmin=lifetime_fine.min(), vmax=lifetime_fine.max()) if lifetime_fine.size else None

mesh = ax.pcolormesh(rho_fine, M_fine, lifetime_fine, shading="auto", norm=norm, cmap="viridis")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"Background density $\rho$ [g/cc]")
ax.set_ylabel("PBH initial mass $M_{PBH}$ [g]")
ax.set_title("PBH lifetime (RegularGridInterpolator - Cubic)")

cbar = fig.colorbar(mesh, ax=ax)
cbar.set_label("Lifetime [Gyr]")

plt.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '../../blackhawk_v2.3/results/PBH_evolution/M1.00e+13/rho1.00e+06/dts.txt'